# Introduction

Welcome to this comprehensive guide on **binary classification** with the **Spaceship Titanic** dataset. The objective is to predict whether a passenger was transported to an alternate dimension during the Spaceship Titanic's collision with a spacetime anomaly.

*We will cover:*
* Exploratory and preparation Data
* Encoding, Scaling and Preprocessing
* Training Machine Learning Models
* Cross Validation and Ensembling Predictions

# Data Exploration and Preparation

In [ ]:
import numpy as np 
import pandas as pd 


upload dataset ???????????? Train ??? Test model ???????? index ???? PassengerId

In [ ]:

train = pd.read_csv('../input/spaceship-titanic/train.csv', index_col='PassengerId')
test = pd.read_csv('../input/spaceship-titanic/test.csv', index_col='PassengerId')

# Shape and preview
print('Train set shape:', train.shape)
print('Test set shape:', test.shape)
train.head()

*Feature descriptions:*
> * **PassengerId** - A unique Id for each passenger. Each Id takes the form gggg_pp where gggg indicates a group the passenger is travelling with and pp is their number within the group. People in a group are often family members, but not always.
> * **HomePlanet** - The planet the passenger departed from, typically their planet of permanent residence.
> * **CryoSleep** - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
> * **Cabin** - The cabin number where the passenger is staying. Takes the form deck/num/side, where side can be either P for Port or S for Starboard.
> * **Destination** - The planet the passenger will be debarking to.
> * **Age** - The age of the passenger.
> * **VIP** - Whether the passenger has paid for special VIP service during the voyage.
> * **RoomService**, **FoodCourt**, **ShoppingMall**, **Spa**, **VRDeck** - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
> * **Name** - The first and last names of the passenger.
> * **Transported** - Whether the passenger was transported to another dimension. This is the target, the column you are trying to predict.

?? column "Name" ??????????????? train ??? test

In [ ]:
train.drop('Name', axis=1, inplace=True)
test.drop('Name', axis=1, inplace=True)

????????? False ???? 0 ???????????? True ???? 1 ?? column Transported

In [ ]:
train['Transported'].replace(False, 0, inplace=True)
train['Transported'].replace(True, 1, inplace=True)

???column Cabin ????3column??? deck num side

In [ ]:
train[['deck','num', 'side']] = train['Cabin'].str.split('/', expand=True)
test[['deck','num', 'side']] = test['Cabin'].str.split('/', expand=True)

train.drop('Cabin', axis=1, inplace=True)
test.drop('Cabin', axis=1, inplace=True)

????? column SumSpends ?????????????????????? ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

In [ ]:
col_to_sum = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

train['SumSpends'] = train[col_to_sum].sum(axis=1)
test['SumSpends'] = test[col_to_sum].sum(axis=1)

In [ ]:
train

In [ ]:
test

print ???????????????????????????????????? Dataset ????????????????? 3 ????????? dtypes: float64(7), int64(1), object(7)

In [ ]:
train.info()

???????????????????????? Missing Value ??????? ?????????????????????

In [ ]:
train.isna().sum()

In [ ]:
test.isna().sum()

In [ ]:
null_cols = train.isnull().sum().sort_values(ascending=False)
null_cols = list(null_cols[null_cols>1].index)
null_cols

# Cleaning, Encoding, Scaling and Preprocessing

object_cols ??????????????? object ??? category
numeric_cols ??????????????? float64 




In [ ]:
object_cols = [col for col in train.columns if train[col].dtype == 'object' or train[col].dtype == 'category']
numeric_cols = [col for col in train.columns if train[col].dtype == 'float64']

print(f'Object cols -- {object_cols}')
print(f'Numeric cols -- {numeric_cols}')

??? object_cols ??? train ??? test ????????????? category

In [ ]:
train[object_cols] = train[object_cols].astype('category')
test[object_cols] = test[object_cols].astype('category')

Encoding (??????????) ??? categorical ??? object ???? string ?????????? float ??????????

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

oc = OrdinalEncoder()

df_for_encode = pd.concat([train, test])

df_for_encode[object_cols] = df_for_encode[object_cols].astype('category')

df_for_encode[object_cols] = oc.fit_transform(df_for_encode[object_cols])

del train, test

train = df_for_encode.iloc[:8693, :]
test = df_for_encode.iloc[8693: , :]

del df_for_encode

test.drop('Transported', inplace=True, axis=1)

?? column Transported ?????? test set ?????????????????????????????? Transported

In [ ]:
test

???????????????? missing value ??????? mean ??? column ????

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


ct = ColumnTransformer([("imp", SimpleImputer(strategy='mean'), null_cols)])
    
train[null_cols] = ct.fit_transform(train[null_cols])
test[null_cols] = ct.fit_transform(test[null_cols])

# Prepareing dataset for modeling

??????????????????????? ???????????? X = train dataset ??? y ??? Transported ???????????? train 70% test 30% ????????? train dataset ???????????????????? 

In [ ]:
X = train.copy()
y = X.pop('Transported')

from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=23)

??????? catboost ??????????????? CatBoostClassifier

In [ ]:
!pip3 install catboost


# Modeling Cross Validation and Ensembling Predictions 7 model



1.  RandomForestClassifier
2.  AdaBoostClassifier
3.  LGBMClassifier
4.  XGBClassifier
5.  CatBoostClassifier
6.  KNeighborsClassifier
7.  DecisionTreeClassifier




*  import ????????????????????????????? training 
*  algorithm ?????????????????????????????????? model ??????????????? fit ??? cross_val_score ?????????????? predict_and_acc




In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier
model_list= {}
def predict_and_acc(model, verbose=None):
    
    if verbose == None:
        model = model()
        model.fit(X_train, y_train)
        predict = model.predict(X_test)
        cvs = cross_val_score(model, X, y, cv=4)
        print(f'The accuracy score of {str(model)} is {float(accuracy_score(y_test, predict))}')
        print(f'The cross validation of {str(model)} is:{cvs} with mean of {cvs.mean()}')
    else:
        model = model(verbose=verbose)
        model.fit(X_train, y_train)
        predict = model.predict(X_test)
        cvs = cross_val_score(model, X, y, cv=4)
        print(f'The accuracy score of {str(model)} is {float(accuracy_score(y_test, predict))}')
        print(f'The cross validation of {str(model)} is:{cvs} with mean of {cvs.mean()}')



In [ ]:
predict_and_acc(RandomForestClassifier, None)


In [ ]:
predict_and_acc(AdaBoostClassifier)


In [ ]:
predict_and_acc(LGBMClassifier)

In [ ]:
predict_and_acc(CatBoostClassifier, verbose=False)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
predict_and_acc(KNeighborsClassifier)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
predict_and_acc(DecisionTreeClassifier)


??? Backward Feature Selection ??? CatBoostClassifier (????????????? model ?????????????????????)

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector

model_fs = CatBoostClassifier(verbose=False)
sf = SequentialFeatureSelector(model_fs, scoring='accuracy', direction = 'backward')
sf.fit(X,y)

?? feature ?????????????????????????????? (???????????????????????????? feature ???????????????)

In [ ]:
best_features = list(sf.get_feature_names_out())
best_features

?????????????????????????? best_features ??? X (train set)

In [ ]:
model = CatBoostClassifier(verbose=False, eval_metric='Accuracy')
model.fit(X[best_features], y)
prediction = model.predict(test[best_features])

# Ensembling Predictions

???????????????????????????? 0 = false ??? 1 = True


In [ ]:
final = pd.DataFrame()
final.index = test.index
final['Transported'] = prediction
final['Transported'].replace(0, False, inplace=True)
final['Transported'].replace(1, True, inplace=True)
final

save ????????????????????

In [ ]:
final.to_csv('submission.csv')